In [10]:
import adapters
import torch
from transformers import AutoModel

from transformers import AutoTokenizer

torch.manual_seed(42)

model_name = "thomas-sounack/BioClinical-ModernBERT-base"

model1 = AutoModel.from_pretrained(model_name)
torch.manual_seed(42)
model2 = AutoModel.from_pretrained(model_name)
torch.manual_seed(42)
model3 = AutoModel.from_pretrained(model_name)
torch.manual_seed(42)

adapters.init(model1)
adapters.init(model2)
adapters.init(model3)

In [2]:
torch.manual_seed(42)
model1.add_adapter("modernbert-adapter")
model1.train_adapter("modernbert-adapter")

model1.active_head = None

There are adapters available but none are activated for the forward pass.


In [3]:
torch.manual_seed(42)
model2.add_adapter("modernbert-adapter")
model2.set_active_adapters("modernbert-adapter")
model2.train_adapter("modernbert-adapter")


model2.active_head = None

There are adapters available but none are activated for the forward pass.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
# sample tokenize text
inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")


In [5]:
# Test in eval mode
model1.eval()
model2.eval()
model3.eval()

ModernBertModel(
  (embeddings): ModernBertEmbeddings(
    (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (layers): ModuleList(
    (0): ModernBertEncoderLayer(
      (attn_norm): Identity()
      (attn): ModernBertAttention(
        (Wqkv): LoRAMergedLinear(
          in_features=768, out_features=2304, bias=True
          (shared_parameters): ModuleDict()
          (loras): ModuleDict()
        )
        (rotary_emb): ModernBertRotaryEmbedding()
        (Wo): Linear(in_features=768, out_features=768, bias=False)
        (out_drop): Identity()
      )
      (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): ModernBertMLP(
        (Wi): LoRALinearTorch(
          in_features=768, out_features=2304, bias=False
          (shared_parameters): ModuleDict()
          (loras): ModuleDict()
        )
        (act): GELUActivation()
        (

In [6]:
with torch.no_grad():
    out1 = model1(**inputs).last_hidden_state
    out2 = model2(**inputs).last_hidden_state
    out3 = model3(**inputs).last_hidden_state

print("Out1 vs Out3:", torch.allclose(out1, out3, atol=1e-6))
print("Max diff 1-3:", torch.abs(out1 - out3).max().item())
print("Max diff 1-2:", torch.abs(out1 - out2).max().item())
print("Max diff 2-3:", torch.abs(out2 - out3).max().item())

Out1 vs Out3: False
Max diff 1-3: 2.1763763427734375
Max diff 1-2: 2.581493377685547
Max diff 2-3: 1.2031116485595703


In [8]:
model1.adapter_summary()

'================================================================================\nName                     Architecture         #Param      %Param  Active   Train\n--------------------------------------------------------------------------------\nmodernbert-adapter       bottleneck        1,639,968       1.100       1       1\n--------------------------------------------------------------------------------\nFull model                               149,064,960     100.000               0\n================================================================================'

'================================================================================\nName                     Architecture         #Param      %Param  Active   Train\n--------------------------------------------------------------------------------\nmodernbert-adapter       bottleneck        1,639,968       1.100       1       1\n--------------------------------------------------------------------------------\nFull model                               149,064,960     100.000               0\n================================================================================'

In [8]:
with torch.no_grad():
    print(model1(**inputs).last_hidden_state)

tensor([[[ 0.5808, -0.8078, -1.1482,  ..., -1.5763, -0.4304, -1.1711],
         [ 1.3489, -0.2959,  0.7184,  ...,  0.4295, -0.1065, -0.8241],
         [ 0.0790, -0.0710,  0.1452,  ...,  0.2508,  0.0526,  0.1280],
         ...,
         [-0.2958, -0.3950, -0.0099,  ..., -1.1235, -0.0278,  0.1742],
         [ 1.6550, -0.4287,  1.1616,  ..., -0.9369, -0.0695, -0.8149],
         [ 0.2949,  0.0218,  0.1452,  ..., -0.0615,  0.0848, -0.2283]]])


In [9]:
with torch.no_grad():
    print(model2(**inputs).last_hidden_state)

tensor([[[ 0.6066, -0.9421, -1.1954,  ..., -1.6653, -0.3690, -1.2557],
         [ 1.3107, -0.1502,  0.6786,  ...,  0.3151,  0.1192, -0.7527],
         [ 0.0723, -0.0701,  0.1412,  ...,  0.2556,  0.0578,  0.1204],
         ...,
         [-0.5251, -0.1510,  0.0215,  ..., -1.0917, -0.0470,  0.1045],
         [ 1.3316, -0.3380,  1.1797,  ..., -0.7535, -0.2395, -0.8178],
         [ 0.2912,  0.0031,  0.1433,  ..., -0.0453,  0.0871, -0.2312]]])


In [10]:
with torch.no_grad():
    print(model3(**inputs).last_hidden_state)

tensor([[[ 0.3165, -0.6964, -1.2377,  ..., -1.9806, -0.7944, -1.1640],
         [ 0.9658,  0.2135,  0.8438,  ...,  0.5546, -0.2069, -0.4024],
         [ 0.1268,  0.0763,  0.2183,  ...,  0.1444, -0.0425,  0.0553],
         ...,
         [-0.4226, -0.1760,  0.1694,  ..., -1.2321, -0.3938,  0.2824],
         [ 1.2833, -0.2579,  1.4540,  ..., -0.7295, -0.3196, -0.4442],
         [ 0.2924, -0.0801,  0.1431,  ...,  0.0098,  0.0120, -0.2714]]])


In [11]:
model1.adapter_summary

<bound method ModelAdaptersMixin.adapter_summary of ModernBertModel(
  (embeddings): ModernBertEmbeddings(
    (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (drop): Dropout(p=0.0, inplace=False)
  )
  (layers): ModuleList(
    (0): ModernBertEncoderLayer(
      (attn_norm): Identity()
      (attn): ModernBertAttention(
        (Wqkv): LoRAMergedLinear(
          in_features=768, out_features=2304, bias=True
          (shared_parameters): ModuleDict()
          (loras): ModuleDict()
        )
        (rotary_emb): ModernBertRotaryEmbedding()
        (Wo): Linear(in_features=768, out_features=768, bias=False)
        (out_drop): Identity()
      )
      (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): ModernBertMLP(
        (Wi): LoRALinearTorch(
          in_features=768, out_features=2304, bias=False
          (shared_parameters): ModuleDict()
          (loras): ModuleDict()

In [10]:
model1(**inputs)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.0386,  0.1001,  0.0043,  ..., -0.1316,  0.1856, -0.0739],
         [ 0.1390,  0.2635,  0.2498,  ...,  0.1015, -0.2526, -0.0192],
         [-0.1463, -0.0636,  0.1745,  ...,  0.0325,  0.2084, -0.0346],
         ...,
         [-0.0950,  0.0465, -0.0367,  ...,  0.0288, -0.3683, -0.0918],
         [-0.1564,  0.1809,  0.0972,  ..., -0.0816, -0.2100,  0.1608],
         [-0.0151, -0.0331, -0.0042,  ..., -0.0662, -0.0940, -0.0240]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-7.3076e-01, -5.0703e-02, -7.6215e-01,  4.4892e-02,  7.4259e-01,
          7.1840e-01, -5.7694e-02,  1.4279e-02, -5.4314e-01,  1.0153e-02,
          5.5920e-02, -1.9639e-01, -2.4960e-01, -8.3659e-02, -4.8643e-03,
          1.2040e-01, -2.6589e-02,  2.7603e-02,  7.1715e-02,  1.0000e-01,
         -4.4516e-02, -2.5705e-02,  1.3656e-01, -2.5959e-02,  9.1815e-01,
          8.1257e-01,  1.9073e-01,  1.6416e-02, -9.9729e-01, -4.654

In [11]:
import torch
import adapters
from transformers import AutoModel, AutoTokenizer

# Control randomness
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model1 = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
model2 = AutoModel.from_pretrained("answerdotai/ModernBERT-base")
model3 = AutoModel.from_pretrained("answerdotai/ModernBERT-base")

adapters.init(model1)
adapters.init(model2)
adapters.init(model3)

model1.add_adapter("modernbert-adapter")
model1.train_adapter("modernbert-adapter")

model2.add_adapter("modernbert-adapter")
model2.train_adapter("modernbert-adapter")
model2.set_active_adapters("modernbert-adapter")

# IMPORTANT: Set to eval mode
model1.eval()
model2.eval()
model3.eval()

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

# Use no_grad for deterministic inference
with torch.no_grad():
    out1 = model1(**inputs).last_hidden_state
    out2 = model2(**inputs).last_hidden_state
    out3 = model3(**inputs).last_hidden_state

# Check
print("Model1 == Model3:", torch.allclose(out1, out3, atol=1e-5))

There are adapters available but none are activated for the forward pass.
There are adapters available but none are activated for the forward pass.


Model1 == Model3: False


In [12]:
print("Model1 == Model2:", torch.allclose(out1, out2, atol=1e-5))

Model1 == Model2: False


In [11]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')
